# Módulo 0.5 — Do sequenciador à matriz de contagens

**Workshop de RNA-seq · mieloma múltiplo**
Este módulo roda **antes do Dia 1**.

---

O Dia 1 começa com uma matriz de contagens pronta. Mas ela não nasce assim. Alguém
pegou os arquivos que saíram do sequenciador e rodou cinco etapas até chegar nela.

Aqui você vai rodar essas cinco etapas. De verdade — não é simulação, não é vídeo.
Os comandos são os mesmos que um serviço de bioinformática usaria, e você vai ver
cada arquivo intermediário.

Leva **10 a 15 minutos**.

## Por que levedura, e não mieloma?

Porque os FASTQ do MMRF-CoMMpass são de **acesso controlado**, via dbGaP. A camada
aberta do GDC entrega contagens, não leituras.

Isso já é a primeira lição: **o nível de acesso ao dado determina em que ponto do
fluxo você consegue entrar.** Guarde isso — volta no Dia 2, quando o sexo dos
pacientes também não estiver disponível.

Vamos usar 4 amostras de *Saccharomyces cerevisiae* do conjunto de teste do nf-core
(GSE110004), subamostradas para poucos MB. O genoma da levedura tem 12 Mb contra
3 Gb do humano — é o que permite construir o índice em um minuto em vez de uma hora.

## O caminho

```
FASTQ  →  QC  →  trimagem  →  alinhamento  →  BAM  →  contagem  →  MATRIZ
 (1)      (2)      (3)          (4)          (5)      (6)          ↓
                                                            aqui começa o Dia 1
```

> ⚠️ **Ferramentas de linha de comando, não de R ou Python.** FastQC, fastp,
> HISAT2, samtools e featureCounts são programas independentes. Este notebook só
> os chama. É por isso que esta etapa roda no Colab, que é Linux, e não na sua
> máquina Windows.

---
## 1 · Instalar as ferramentas

Cinco programas, direto do repositório do Ubuntu. Leva 1 a 2 minutos.

In [ ]:
%%bash
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq fastqc fastp hisat2 samtools subread > /dev/null 2>&1
echo "instalação concluída"

In [ ]:
# Conferir se cada ferramenta responde. Se alguma falhar, o resto não roda —
# melhor descobrir agora do que no meio do alinhamento.
import shutil
import subprocess
import sys

FERRAMENTAS = {
    "fastqc":       ["fastqc", "--version"],
    "fastp":        ["fastp", "--version"],
    "hisat2":       ["hisat2", "--version"],
    "hisat2-build": ["hisat2-build", "--version"],
    "samtools":     ["samtools", "--version"],
    "featureCounts":["featureCounts", "-v"],
}

print("Verificando ferramentas de bioinformática...\n")

faltando = []
erros = []

for nome, cmd in FERRAMENTAS.items():
    if shutil.which(cmd[0]) is None:
        faltando.append(nome)
        print(f"  ❌ {nome:<14} não encontrado")
        continue

    try:
        r = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            errors='replace',
            timeout=10
        )

        versao = (r.stdout + r.stderr).strip().split("\n")[0][:60]
        print(f"  ✅ {nome:<14} {versao}")

    except subprocess.TimeoutExpired:
        erros.append(f"{nome} (timeout)")
        print(f"  ⚠️  {nome:<14} timeout ao executar")
    except Exception as e:
        erros.append(f"{nome} ({str(e)})")
        print(f"  ⚠️  {nome:<14} erro: {type(e).__name__}")

print("\n" + "="*60)

if faltando or erros:
    msg = []
    if faltando:
        msg.append(f"❌ Não instaladas: {', '.join(faltando)}")
    if erros:
        msg.append(f"⚠️  Erros ao verificar: {', '.join(erros)}")

    print("\n".join(msg))
    print("\n❌ Não é seguro prosseguir. Instale as ferramentas faltantes.")
    sys.exit(1)
else:
    print("✅ Todas as ferramentas foram encontradas e respondem.")
    print("✅ Seguro prosseguir com a análise.")
    print("\n" + "="*60)

---
## 2 · Baixar os dados

Quatro amostras, o genoma e a anotação. Tudo do repositório público de dados de
teste do nf-core.

| Amostra | Grupo | O que é |
|---|---|---|
| SRR6357070 | WT | levedura selvagem, réplica 1 |
| SRR6357072 | WT | levedura selvagem, réplica 2 |
| SRR6357073 | RAP1 | linhagem RAP1, sem indução |
| SRR6357076 | RAP1 | linhagem RAP1, 30 min de auxina |

Usamos só a leitura R1 de cada uma (*single-end*). O fluxo com leitura pareada é
o mesmo, com um arquivo a mais por amostra.

In [ ]:
%%bash
set -e
mkdir -p dados/fastq dados/ref

BASE=https://raw.githubusercontent.com/nf-core/test-datasets/rnaseq

for SRR in SRR6357070 SRR6357072 SRR6357073 SRR6357076; do
  if [ ! -s dados/fastq/${SRR}.fastq.gz ]; then
    wget -q -O dados/fastq/${SRR}.fastq.gz \
      ${BASE}/testdata/GSE110004/${SRR}_1.fastq.gz
  fi
done

# genoma e anotação
[ -s dados/ref/genoma.fa ] || \
  wget -q -O dados/ref/genoma.fa https://github.com/nf-core/test-datasets/raw/rnaseq/reference/genome.fasta
[ -s dados/ref/genes.gtf ] || {
  wget -q -O dados/ref/genes.gtf.gz https://github.com/nf-core/test-datasets/raw/rnaseq/reference/genes.gtf.gz
  gunzip -f dados/ref/genes.gtf.gz
}

echo "=== arquivos baixados ==="
ls -lh dados/fastq/ dados/ref/

In [ ]:
# Trava: arquivo truncado aqui vira erro incompreensível no alinhamento.
from pathlib import Path
import gzip

AMOSTRAS = {"SRR6357070": "WT", "SRR6357072": "WT",
            "SRR6357073": "RAP1", "SRR6357076": "RAP1"}

problemas = []
for srr in AMOSTRAS:
    p = Path(f"dados/fastq/{srr}.fastq.gz")
    if not p.exists() or p.stat().st_size < 10_000:
        problemas.append(f"{srr}: ausente ou pequeno demais"); continue
    try:                                     # o gzip abre até o fim?
        with gzip.open(p, "rt") as fh:
            n = sum(1 for _ in fh)
        if n % 4:
            problemas.append(f"{srr}: {n} linhas, não é múltiplo de 4 — truncado")
        else:
            print(f"  ✅ {srr}  {p.stat().st_size/1e6:.1f} MB  {n//4:,} leituras".replace(",", "."))
    except Exception as e:
        problemas.append(f"{srr}: {type(e).__name__} ao descompactar")

for nome, minimo in [("dados/ref/genoma.fa", 1e5), ("dados/ref/genes.gtf", 1e4)]:
    p = Path(nome)
    if not p.exists() or p.stat().st_size < minimo:
        problemas.append(f"{nome}: ausente ou incompleto")
    else:
        print(f"  ✅ {nome}  {p.stat().st_size/1e6:.2f} MB")

if problemas:
    raise RuntimeError("Download incompleto:\n  " + "\n  ".join(problemas) +
                       "\n\nRode a célula anterior de novo.")
print("\nTodos os arquivos íntegros.")

---
## 3 · Anatomia de um FASTQ

Este é o arquivo que sai do sequenciador. Antes de processar, **olhe**.

In [ ]:
!zcat dados/fastq/SRR6357070.fastq.gz | head -8

### O que você está vendo

São **quatro linhas por leitura**, sempre nessa ordem:

| Linha | Conteúdo |
|---|---|
| 1 | identificador, começa com `@` |
| 2 | a sequência de bases |
| 3 | separador, começa com `+` |
| 4 | a qualidade de cada base, **um caractere por base** |

A linha 4 é a que quase ninguém entende. Cada caractere é um número codificado em
ASCII — o **Phred score**. A conta é:

$$Q = -10 \times \log_{10}(P_{\text{erro}})$$

Um `Q30` significa 1 chance em 1.000 de a base estar errada. Um `Q20`, 1 em 100.

A célula abaixo decodifica a qualidade da primeira leitura, caractere por caractere.

In [ ]:
import gzip

with gzip.open("dados/fastq/SRR6357070.fastq.gz", "rt") as fh:
    ident, seq, _, qual = [next(fh).rstrip() for _ in range(4)]

print(f"identificador : {ident}")
print(f"comprimento   : {len(seq)} bases\n")
print("  base  caractere  Phred   prob. de erro")
print("  " + "-" * 44)
for i in range(min(12, len(seq))):
    q = ord(qual[i]) - 33          # Illumina usa offset 33
    p = 10 ** (-q / 10)
    print(f"   {seq[i]}        {qual[i]}        {q:>3}     1 em {1/p:>10,.0f}".replace(",", "."))

phred = [ord(ch) - 33 for ch in qual]
print(f"\n  Phred médio desta leitura : {sum(phred)/len(phred):.1f}")
print(f"  menor Phred               : {min(phred)}")
print(f"  bases com Q >= 30         : {100*sum(q >= 30 for q in phred)/len(phred):.0f}%")

print("""
💬 Repare que a qualidade costuma cair no FIM da leitura. Isso é característico
   do sequenciamento por síntese: a cada ciclo, uma fração das moléculas do
   cluster fica fora de fase, e o sinal vai perdendo nitidez.""")

---
## 4 · Controle de qualidade — FastQC

O FastQC lê o FASTQ inteiro e produz um relatório com uma dúzia de gráficos. Não
corrige nada: só mostra.

In [ ]:
%%bash
mkdir -p qc/fastqc
fastqc -q -t 2 -o qc/fastqc dados/fastq/*.fastq.gz
ls qc/fastqc/

In [ ]:
# O relatório é um HTML, mas dentro do .zip vem também o resumo em texto e os
# gráficos em PNG. Vamos ler os dois.
import zipfile, glob
from pathlib import Path
from IPython.display import display, Image, Markdown

resumos = {}
for z in sorted(glob.glob("qc/fastqc/*_fastqc.zip")):
    nome = Path(z).stem.replace("_fastqc", "")
    with zipfile.ZipFile(z) as zf:
        raiz = zf.namelist()[0].split("/")[0]
        txt = zf.read(f"{raiz}/summary.txt").decode()
        resumos[nome] = [l.split("\t")[:2] for l in txt.strip().split("\n")]

modulos = [m for _, m in resumos[list(resumos)[0]]]
print(f"{'módulo':<38}" + "".join(f"{n[-4:]:>8}" for n in resumos))
print("-" * (38 + 8 * len(resumos)))
simbolo = {"PASS": "  ✅", "WARN": "  ⚠️", "FAIL": "  ❌"}
for i, m in enumerate(modulos):
    linha = f"{m[:37]:<38}"
    for nome in resumos:
        linha += f"{simbolo.get(resumos[nome][i][0], '  ?'):>8}"
    print(linha)

print("""
💬 ⚠️ e ❌ no FastQC NÃO significam dado ruim. Os limites são genéricos, pensados
   para sequenciamento de genoma. Em RNA-seq, três módulos falham quase sempre —
   e por motivos biológicos, não técnicos:

   · Per base sequence content  — o início da leitura tem viés dos primers
                                   aleatórios usados na síntese do cDNA
   · Sequence duplication levels — gene muito expresso gera leituras idênticas.
                                   Isso é sinal, não artefato
   · Overrepresented sequences   — mesma coisa: rRNA e transcritos abundantes

   Olhe o relatório, não o semáforo.""")

In [ ]:
# O gráfico que importa: qualidade por posição na leitura.
with zipfile.ZipFile("qc/fastqc/SRR6357070_fastqc.zip") as zf:
    raiz = zf.namelist()[0].split("/")[0]
    for img in ["per_base_quality.png", "per_sequence_quality.png"]:
        try:
            with open(img, "wb") as fh:
                fh.write(zf.read(f"{raiz}/Images/{img}"))
            display(Markdown(f"**{img}**"))
            display(Image(img))
        except KeyError:
            print(f"({img} não está neste relatório)")

---
## 5 · Trimagem — fastp

Remove adaptadores e apara as pontas de baixa qualidade.

> ⚠️ **Trimagem agressiva é prática superada.** Durante anos se cortou tudo abaixo
> de Q30. Hoje se sabe que isso **piora** a quantificação: você joga fora leituras
> boas e enviesa a cobertura. Os alinhadores modernos fazem *soft-clipping* — eles
> ignoram as pontas ruins sozinhos.
>
> Williams *et al.* (2016), *BMC Bioinformatics*, mostrou isso com números.
>
> Aqui usamos parâmetros conservadores de propósito. Repare no fim quantas
> leituras foram descartadas: pouquíssimas.

In [ ]:
%%bash
set -e
mkdir -p limpo qc/fastp

for SRR in SRR6357070 SRR6357072 SRR6357073 SRR6357076; do
  fastp \
    -i dados/fastq/${SRR}.fastq.gz \
    -o limpo/${SRR}.fastq.gz \
    --qualified_quality_phred 15 \
    --length_required 25 \
    --detect_adapter_for_pe \
    --json qc/fastp/${SRR}.json \
    --html qc/fastp/${SRR}.html \
    --thread 2 2> /dev/null
done
echo "trimagem concluída"

In [ ]:
# O fastp grava um JSON com o antes e o depois. É aí que se vê o efeito real.
import json as _json

print(f"{'amostra':<14}{'leituras antes':>16}{'depois':>12}{'descartado':>13}{'Q30 antes':>12}{'Q30 depois':>12}")
print("-" * 79)
for srr in AMOSTRAS:
    with open(f"qc/fastp/{srr}.json") as fh:
        j = _json.load(fh)
    antes, depois = j["summary"]["before_filtering"], j["summary"]["after_filtering"]
    perda = 100 * (1 - depois["total_reads"] / antes["total_reads"])
    print(f"{srr:<14}{antes['total_reads']:>16,}{depois['total_reads']:>12,}"
          f"{perda:>12.1f}%{100*antes['q30_rate']:>11.1f}%{100*depois['q30_rate']:>11.1f}%"
          .replace(",", "."))

print("""
💬 A perda é pequena, e o Q30 sobe pouco. É exatamente esse o ponto: com dados
   modernos, a trimagem quase não muda nada. Ela existe para o caso em que há
   adaptador de verdade — e é o fastp quem detecta isso, não você.""")

---
## 6 · Alinhamento — HISAT2

Cada leitura precisa ser localizada no genoma. O desafio do RNA-seq é que uma
leitura pode **atravessar uma junção de éxons** — metade dela cai num éxon e a
outra metade em outro, com um íntron de milhares de bases no meio.

Alinhador de DNA não sabe fazer isso. HISAT2 e STAR sabem: são *splice-aware*.

O primeiro passo é construir o índice do genoma. Para a levedura leva menos de um
minuto. Para o genoma humano levaria mais de uma hora e uns 200 GB de RAM com o
STAR — é por isso que ninguém refaz o índice: você baixa pronto.

In [ ]:
%%bash
set -e
mkdir -p indice
if [ ! -s indice/levedura.1.ht2 ]; then
  hisat2-build -p 2 dados/ref/genoma.fa indice/levedura > /dev/null 2>&1
fi
echo "=== índice construído ==="
ls -lh indice/ | head -5

echo ""
echo "=== o que tem no genoma de referência ==="
grep "^>" dados/ref/genoma.fa | head
echo ""
grep -c "^>" dados/ref/genoma.fa | xargs echo "sequências:"

In [ ]:
%%bash
set -e
mkdir -p bam qc/hisat2

for SRR in SRR6357070 SRR6357072 SRR6357073 SRR6357076; do
  hisat2 -p 2 -x indice/levedura -U limpo/${SRR}.fastq.gz \
    --summary-file qc/hisat2/${SRR}.txt \
    2> /dev/null \
  | samtools sort -@ 2 -o bam/${SRR}.bam -
  samtools index bam/${SRR}.bam
done

echo "=== BAM gerados ==="
ls -lh bam/*.bam

### A taxa de alinhamento é o primeiro número que se olha

Abaixo de 70% em RNA-seq há algo errado: contaminação, genoma de referência
trocado, ou adaptador que não foi removido.

In [ ]:
import re

print(f"{'amostra':<14}{'leituras':>12}{'alinhadas':>12}{'taxa':>9}   situação")
print("-" * 62)
for srr in AMOSTRAS:
    txt = open(f"qc/hisat2/{srr}.txt").read()
    total = int(re.search(r"(\d+) reads; of these", txt).group(1))
    taxa  = float(re.search(r"([\d.]+)% overall alignment rate", txt).group(1))
    alinhadas = round(total * taxa / 100)
    marca = "✅ boa" if taxa >= 70 else ("⚠️ baixa — investigue" if taxa >= 50
                                        else "❌ algo está errado")
    print(f"{srr:<14}{total:>12,}{alinhadas:>12,}{taxa:>8.1f}%   {marca}".replace(",", "."))

print("""
💬 O que derruba a taxa de alinhamento, em ordem de frequência:
   1. genoma de referência errado (organismo ou versão)
   2. adaptador não removido — a leitura não casa com nada
   3. contaminação por outro organismo
   4. rRNA em excesso, se a biblioteca não foi depletada""")

---
## 7 · O que é um BAM

BAM é a versão binária e comprimida do SAM. Uma linha por alinhamento, com onde a
leitura caiu, quão bem casou, e a leitura em si.

Você nunca abre um BAM direto — usa o `samtools` para traduzir.

In [ ]:
%%bash
echo "=== três alinhamentos, em formato legível ==="
samtools view bam/SRR6357070.bam | head -3
echo ""
echo "=== resumo do arquivo ==="
samtools flagstat bam/SRR6357070.bam

### Lendo uma linha do SAM

As primeiras colunas são o essencial:

| Coluna | O que é |
|---|---|
| 1 | nome da leitura |
| 2 | *flag* — bits que dizem se alinhou, se é reversa, se é duplicada |
| 3 | em que cromossomo caiu |
| 4 | em que posição |
| 5 | MAPQ — confiança do alinhamento |
| 6 | CIGAR — o mapa do alinhamento |

O **CIGAR** é o campo mais informativo. Um `50M` significa 50 bases casadas em
sequência. Um `30M2000N20M` significa 30 bases num éxon, 2.000 bases puladas
(o íntron), e 20 bases no éxon seguinte.

**O `N` é a assinatura de uma junção de éxons.** É o que só um alinhador
*splice-aware* produz.

In [ ]:
# Quantas leituras atravessaram junções? Procuramos o N no CIGAR.
import subprocess, collections

r = subprocess.run(["samtools", "view", "bam/SRR6357070.bam"],
                   capture_output=True, text=True)
linhas = r.stdout.strip().split("\n")

cigars = [l.split("\t")[5] for l in linhas if len(l.split("\t")) > 5]
com_juncao = [c for c in cigars if "N" in c]

print(f"alinhamentos            : {len(cigars):,}".replace(",", "."))
print(f"atravessando junção (N) : {len(com_juncao):,}".replace(",", ".")
      + f"  ({100*len(com_juncao)/max(len(cigars),1):.1f}%)")

if com_juncao:
    print("\nexemplos de CIGAR com junção:")
    for c in com_juncao[:5]:
        print("   ", c)
else:
    print("""
   Nenhuma junção nesta amostra. É esperado: a levedura tem pouquíssimos íntrons
   — cerca de 300 genes em 6.000. No humano, quase todo gene tem, e a fração de
   leituras com N passa de 30%.""")

print("\ndistribuição de MAPQ (confiança do alinhamento):")
mapq = collections.Counter(int(l.split("\t")[4]) for l in linhas if len(l.split("\t")) > 4)
for q, n in sorted(mapq.items(), reverse=True)[:6]:
    print(f"   MAPQ {q:>3} : {n:>7,}".replace(",", ".")
          + f"   {'(único e confiável)' if q >= 30 else '(multimapeada ou ambígua)'}")

---
## 8 · Contagem — featureCounts

Agora a pergunta muda: não é mais "onde essa leitura caiu?", e sim **"quantas
leituras caíram dentro de cada gene?"**

Para responder, é preciso saber onde ficam os genes. É isso que o GTF diz.

In [ ]:
%%bash
echo "=== como é o GTF ==="
grep -v "^#" dados/ref/genes.gtf | head -3
echo ""
echo "=== quantos genes e éxons ==="
awk -F'\t' '$3=="gene"' dados/ref/genes.gtf | wc -l | xargs echo "genes:"
awk -F'\t' '$3=="exon"' dados/ref/genes.gtf | wc -l | xargs echo "éxons:"

In [ ]:
%%bash
set -e
mkdir -p contagens

featureCounts \
  -T 2 \
  -a dados/ref/genes.gtf \
  -t exon \
  -g gene_id \
  -o contagens/matriz.txt \
  bam/*.bam 2> contagens/log.txt

echo "=== resumo da atribuição ==="
cat contagens/matriz.txt.summary

### O que significa cada linha do resumo

| Categoria | O que aconteceu |
|---|---|
| `Assigned` | caiu dentro de um gene — é o que vira contagem |
| `Unassigned_NoFeatures` | alinhou no genoma, mas fora de qualquer gene |
| `Unassigned_Ambiguity` | caiu na sobreposição de dois genes |
| `Unassigned_MultiMapping` | alinhou em vários lugares |

**Você já viu essas categorias antes.** No Dia 1, ao abrir o arquivo STAR do GDC,
as quatro primeiras linhas eram `N_unmapped`, `N_multimapping`, `N_noFeature` e
`N_ambiguous` — e nós as descartamos. São exatamente estas, com outro nome.

Agora você sabe de onde vinham.

---
## 9 · A matriz

É aqui que o Dia 1 começa.

In [ ]:
import pandas as pd
from pathlib import Path

bruta = pd.read_csv("contagens/matriz.txt", sep="\t", comment="#")
print("colunas que o featureCounts devolve:")
print("  ", list(bruta.columns[:6]), "...\n")

# as seis primeiras colunas são anotação; da sétima em diante são as amostras
mat = bruta.set_index("Geneid").iloc[:, 5:]
mat.columns = [Path(c).stem for c in mat.columns]
mat = mat[list(AMOSTRAS)]                       # ordem estável

meta = pd.DataFrame({"grupo": [AMOSTRAS[s] for s in mat.columns]}, index=mat.columns)

print(f"Matriz: {mat.shape[0]:,} genes × {mat.shape[1]} amostras".replace(",", "."))
print(f"\nprofundidade por amostra (leituras atribuídas):")
for s in mat.columns:
    print(f"   {s}  {AMOSTRAS[s]:<5} {mat[s].sum():>10,}".replace(",", "."))

print(f"\ngenes com contagem zero em todas : {(mat.sum(axis=1) == 0).sum():,}".replace(",", "."))
print(f"genes com pelo menos 10 no total : {(mat.sum(axis=1) >= 10).sum():,}".replace(",", "."))

mat.to_csv("matriz_contagens.csv")
meta.to_csv("metadata.csv")
print("\nSalvos: matriz_contagens.csv e metadata.csv")
mat.head(10)

In [ ]:
# Uma olhada rápida: as amostras do mesmo grupo se parecem?
import numpy as np, matplotlib.pyplot as plt

log = np.log2(mat[mat.sum(axis=1) >= 10] + 1)
corr = log.corr(method="spearman")

fig, ax = plt.subplots(figsize=(5.5, 4.6))
im = ax.imshow(corr, cmap="RdYlBu_r", vmin=corr.values.min(), vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels([f"{c[-4:]}\n{AMOSTRAS[c]}" for c in corr.columns], fontsize=9)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels([f"{c[-4:]} {AMOSTRAS[c]}" for c in corr.index], fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.3f}", ha="center", va="center",
                fontsize=8.5, color="black")
ax.set_title("Correlação de Spearman entre amostras", fontsize=11, fontweight="bold")
fig.colorbar(im, fraction=.046)
plt.tight_layout(); plt.show()

print("""
💬 Amostras do mesmo grupo deveriam se parecer mais entre si do que com as do
   outro grupo. Se não se parecem, o problema está no dado ou nos rótulos — e
   nenhuma estatística conserta isso depois.

   Com 4 amostras e um subconjunto do genoma, não espere separação perfeita.""")

---
## 10 · O que você acabou de fazer

```
FASTQ            leituras cruas do sequenciador, 4 linhas por leitura
  ↓ FastQC       diagnóstico — não corrige nada
  ↓ fastp        adaptadores e pontas ruins, com parcimônia
  ↓ HISAT2       localiza cada leitura no genoma, atravessando íntrons
  ↓ samtools     ordena e indexa
  ↓ featureCounts conta quantas leituras por gene
MATRIZ           genes × amostras
```

**Foi exatamente isso que o consórcio MMRF fez com 800 amostras**, com STAR no
lugar do HISAT2 e o genoma humano no lugar do da levedura. O resultado é o arquivo
que o Dia 1 baixa do GDC.

### Três coisas para levar

**1. A matriz de contagens não é o dado bruto.** Ela é o produto de cinco decisões:
qual genoma, qual anotação, qual alinhador, quais parâmetros de trimagem, qual
critério de contagem. Todas afetam o resultado, e nenhuma aparece na matriz.

**2. Por isso o release importa.** Quando o Dia 1 registra "GDC Data Release 46.0",
não é burocracia — é a única forma de outra pessoa refazer estas cinco etapas do
mesmo jeito.

**3. Nível de acesso define onde você entra no fluxo.** Não pudemos usar os FASTQ
do mieloma porque são controlados. Isso não é detalhe administrativo: determina
que análises são possíveis.

---

### 🔧 Experimente

1. Volte à célula do fastp e mude `--qualified_quality_phred` de 15 para 30.
   Rode dali para baixo. Quantas leituras a mais foram descartadas? A taxa de
   alinhamento melhorou o suficiente para compensar?

2. Na célula do featureCounts, troque `-t exon` por `-t gene`. O que muda no
   número de leituras atribuídas, e por quê?

3. Rode `samtools view bam/SRR6357070.bam | awk '$5 < 10' | wc -l` para contar as
   leituras de baixa confiança. Elas entraram na contagem?

---

*Material didático. Dados de teste do nf-core (GSE110004), Saccharomyces
cerevisiae. As ferramentas usadas — FastQC, fastp, HISAT2, samtools, Subread —
são todas de código aberto.*